[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/11-evaluacion-y-pipelines/notebook.ipynb)

# Clase 11 - Evaluacion Robusta y Pipelines
**Objetivo:** evaluar modelos de forma confiable y construir pipelines reutilizables

**Que aprenderemos:**
- Por que un solo train/test split es insuficiente
- Cross-Validation: K-Fold y StratifiedKFold
- Pipelines: encadenar pasos de ML
- GridSearchCV: busqueda de hiperparametros optimos
- Curva de aprendizaje: diagnosticar overfitting/underfitting


## Por que Cross-Validation?

```
PROBLEMA CON UN SOLO SPLIT:
============================

Intento 1 (random_state=42): R2 = 0.85  <-- suerte?
Intento 2 (random_state=7):  R2 = 0.71  <-- mala suerte?
Intento 3 (random_state=99): R2 = 0.79  <-- cual es el real?

El resultado depende de COMO dividiste los datos!

SOLUCION: Cross-Validation
==========================
Divide los datos en K partes (folds) y evalua K veces.
Cada fold actua como test una vez.
El resultado final es el PROMEDIO de los K scores.

  K-Fold (K=5):
  Fold 1: [TEST ][TRAIN][TRAIN][TRAIN][TRAIN] --> score=0.83
  Fold 2: [TRAIN][TEST ][TRAIN][TRAIN][TRAIN] --> score=0.85
  Fold 3: [TRAIN][TRAIN][TEST ][TRAIN][TRAIN] --> score=0.81
  Fold 4: [TRAIN][TRAIN][TRAIN][TEST ][TRAIN] --> score=0.84
  Fold 5: [TRAIN][TRAIN][TRAIN][TRAIN][TEST ] --> score=0.82
                                        Promedio: 0.83 +/- 0.01
```


In [ ]:
# ============================================================
# BLOQUE: Importaciones para Cross-Validation y Pipelines
# ============================================================

# Librerias fundamentales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modelos de clasificacion que usaremos
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Pipeline: encadena pasos de preprocesamiento y modelo
from sklearn.pipeline import Pipeline

# ColumnTransformer: aplica transformaciones distintas a columnas distintas
from sklearn.compose import ColumnTransformer

# Herramientas de evaluacion cruzada
# cross_val_score: evalua modelo con cross-validation
# GridSearchCV: busca los mejores hiperparametros
# StratifiedKFold: K-Fold preservando proporcion de clases
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)

# Preprocesamiento
from sklearn.preprocessing import StandardScaler

# Metricas
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("Librerias cargadas correctamente")


## Seccion 1: Cargar y Preparar los Datos


In [ ]:
# ============================================================
# BLOQUE: Cargar datos y preparar features
# ============================================================

# Cargar dataset de retencion de clientes
df = pd.read_csv('../../datasets/retencion_clientes.csv')
print(f"Dataset cargado: {df.shape}")

# Encodificar variables categoricas
# pd.get_dummies convierte texto a columnas binarias 0/1
df_encoded = pd.get_dummies(df, drop_first=True)

# Identificar columna target
target_col = 'churn'
if target_col not in df_encoded.columns:
    target_col = [c for c in df_encoded.columns if 'churn' in c.lower()][0]

# Separar X e y
X = df_encoded.drop(columns=[target_col])  # todas menos target
y = df_encoded[target_col]                  # solo target

# Separar un conjunto de test FINAL (nunca tocarlo hasta el final)
# Este sera nuestro "holdout" para evaluacion final
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X, y,
    test_size=0.15,   # 15% para evaluacion final
    random_state=42,
    stratify=y
)

print(f"Datos para CV: {X_temp.shape[0]} filas")
print(f"Holdout final: {X_holdout.shape[0]} filas")


## K-Fold Cross Validation Diagram

```
K-FOLD CROSS VALIDATION (K=5)
==============================

Dataset: [  F1  |  F2  |  F3  |  F4  |  F5  ]

Iter 1:  [TEST  |TRAIN |TRAIN |TRAIN |TRAIN ]  score_1
Iter 2:  [TRAIN |TEST  |TRAIN |TRAIN |TRAIN ]  score_2
Iter 3:  [TRAIN |TRAIN |TEST  |TRAIN |TRAIN ]  score_3
Iter 4:  [TRAIN |TRAIN |TRAIN |TEST  |TRAIN ]  score_4
Iter 5:  [TRAIN |TRAIN |TRAIN |TRAIN |TEST  ]  score_5

Score final = mean([s1, s2, s3, s4, s5]) +/- std

VENTAJAS:
  - Todos los datos se usan como train Y como test
  - Resultado mas estable y confiable
  - Detecta si el modelo es consistente
```


In [ ]:
# ============================================================
# BLOQUE: Cross Validation con cross_val_score
# ============================================================

# TODO: Aplica cross_val_score con Decision Tree y 5 folds

# Crear modelo Decision Tree base para evaluar
dt_base = DecisionTreeClassifier(max_depth=5, random_state=42)

# Definir la estrategia de cross-validation
# StratifiedKFold: K-Fold que preserva la proporcion de clases en cada fold
# n_splits: numero de folds (5 es el estandar)
# shuffle: mezclar antes de dividir
cv_strategy = StratifiedKFold(
    n_splits=5,       # 5 folds
    shuffle=True,     # mezclar datos antes de dividir
    random_state=42   # reproducibilidad
)

# TODO: cross_val_score evalua el modelo K veces
# Parametros:
#   estimator: el modelo a evaluar
#   X: features (sin escalar - el tree no lo necesita)
#   y: target
#   cv: estrategia de cross-validation
#   scoring: metrica de evaluacion
cv_scores = ___  # cross_val_score(dt_base, X_temp, y_temp, cv=cv_strategy, scoring='f1')

# Mostrar resultados por fold
print("Scores por fold (F1):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")

print(f"\nMedia:              {cv_scores.mean():.4f}")
print(f"Desviacion estandar: {cv_scores.std():.4f}")
print(f"Intervalo: {cv_scores.mean()-cv_scores.std():.4f} - {cv_scores.mean()+cv_scores.std():.4f}")


In [ ]:
# ============================================================
# BLOQUE: Visualizar Scores de Cross-Validation
# ============================================================

# Crear grafico de barras con los scores de cada fold
fig, ax = plt.subplots(figsize=(8, 5))

# Barras para cada fold
folds = [f'Fold {i}' for i in range(1, len(cv_scores)+1)]
colores = ['steelblue' if s >= cv_scores.mean() else 'salmon' for s in cv_scores]
ax.bar(
    folds,         # nombres en eje x
    cv_scores,     # valores en eje y
    color=colores, # azul si >= media, rojo si < media
    edgecolor='white',
    width=0.6
)

# Linea de la media
ax.axhline(
    y=cv_scores.mean(),    # posicion en eje y
    color='darkblue',      # color
    linewidth=2,           # grosor
    linestyle='--',        # punteada
    label=f'Media = {cv_scores.mean():.4f}'
)

# Etiquetas encima de cada barra
for i, (fold, score) in enumerate(zip(folds, cv_scores)):
    ax.text(i, score + 0.002, f'{score:.3f}', ha='center', fontsize=10)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Cross-Validation: F1 Score por Fold', fontsize=14)
ax.set_ylim(0, 1.1)  # eje y de 0 a 1.1
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Que es un Pipeline?

```
PIPELINE DE ML
===============

SIN Pipeline (peligroso - data leakage posible):

  scaler.fit(X_train) --> X_train_scaled = scaler.transform(X_train)
  modelo.fit(X_train_scaled, y_train)
  X_test_scaled = scaler.transform(X_test)  <-- ok si lo haces bien
  y_pred = modelo.predict(X_test_scaled)
  ... muchos pasos que pueden salir mal

CON Pipeline (seguro y limpio):

  pipe = Pipeline([
    ('scaler', StandardScaler()),        -- Paso 1: escalar
    ('modelo', LogisticRegression())     -- Paso 2: modelo
  ])

  pipe.fit(X_train, y_train)    <-- escala Y entrena automaticamente
  pipe.predict(X_test)          <-- escala Y predice automaticamente

VENTAJAS:
  - No hay data leakage (scaler solo ve X_train)
  - Codigo mas limpio
  - Funciona con GridSearchCV directamente
  - Facil de serializar (guardar con joblib)
```


In [ ]:
# ============================================================
# BLOQUE: Construir Pipeline StandardScaler + LogisticRegression
# ============================================================

# TODO: Construye y evalua un Pipeline con StandardScaler y LogisticRegression

# Pipeline toma una lista de tuplas: (nombre, objeto_transformador_o_modelo)
# Los nombres son importantes para GridSearchCV
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),        # Paso 1: escalar features
    ('modelo', LogisticRegression(       # Paso 2: modelo de clasificacion
        max_iter=1000,
        random_state=42
    ))
])

# Evaluar el pipeline con cross-validation
# El Pipeline aplica el scaler solo en train en cada fold (correcto!)
cv_scores_lr = ___  # cross_val_score(pipe_lr, X_temp, y_temp, cv=cv_strategy, scoring='f1')

print("Pipeline: StandardScaler + LogisticRegression")
print(f"  CV F1 scores: {cv_scores_lr}")
print(f"  Media: {cv_scores_lr.mean():.4f} +/- {cv_scores_lr.std():.4f}")

# Comparar con el Decision Tree sin pipeline
print("\nComparacion:")
print(f"  Decision Tree (sin pipeline): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  LogReg Pipeline:              {cv_scores_lr.mean():.4f} +/- {cv_scores_lr.std():.4f}")


In [ ]:
# ============================================================
# BLOQUE: Entrenar el Pipeline final
# ============================================================

# Dividir X_temp en train y validation para el entrenamiento final
X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

# Entrenar el pipeline completo en datos de entrenamiento
# .fit() aplica todos los pasos del pipeline en orden
pipe_lr.fit(X_train_f, y_train_f)
print("Pipeline entrenado correctamente")

# Predecir en validacion
# .predict() aplica todos los pasos en orden (scaler + modelo)
y_pred_val = pipe_lr.predict(X_val_f)

# Evaluar
acc_val = accuracy_score(y_val_f, y_pred_val)
f1_val  = f1_score(y_val_f, y_pred_val, zero_division=0)
print(f"Accuracy en validacion: {acc_val:.4f}")
print(f"F1 en validacion:       {f1_val:.4f}")


## GridSearchCV: Busqueda de Hiperparametros

```
GRIDSEARCHCV
=============

Objetivo: encontrar la MEJOR combinacion de hiperparametros

param_grid = {
    'max_depth': [3, 5, 10],
    'min_samples_split': [5, 10, 20]
}

GridSearchCV probara TODAS las combinaciones:
  max_depth=3,  min_samples_split=5  --> CV score
  max_depth=3,  min_samples_split=10 --> CV score
  max_depth=3,  min_samples_split=20 --> CV score
  max_depth=5,  min_samples_split=5  --> CV score
  max_depth=5,  min_samples_split=10 --> CV score  <-- mejor?
  max_depth=5,  min_samples_split=20 --> CV score
  max_depth=10, min_samples_split=5  --> CV score
  max_depth=10, min_samples_split=10 --> CV score
  max_depth=10, min_samples_split=20 --> CV score

Total: 3 x 3 = 9 combinaciones x 5 folds = 45 entrenamientos!
```


In [ ]:
# ============================================================
# BLOQUE: GridSearchCV para Decision Tree
# ============================================================

# TODO: Crea y ejecuta GridSearchCV para optimizar el Decision Tree

# Definir el grid de hiperparametros a explorar
# Cada clave es un hiperparametro, cada valor es una lista de opciones
param_grid = {
    'max_depth': [3, 5, 7, 10, None],     # profundidad del arbol (None=sin limite)
    'min_samples_split': [2, 5, 10, 20],  # min muestras para dividir nodo
    'min_samples_leaf': [1, 5, 10]        # min muestras en hoja
}

# Crear el modelo base
dt_grid = DecisionTreeClassifier(random_state=42)

# GridSearchCV: busca los mejores hiperparametros con cross-validation
# estimator: el modelo base
# param_grid: diccionario con los valores a probar
# cv: estrategia de cross-validation
# scoring: metrica a optimizar
# n_jobs=-1: usar todos los nucleos del procesador
# verbose=1: mostrar progreso
grid_search = GridSearchCV(
    estimator=dt_grid,    # modelo a optimizar
    param_grid=param_grid, # combinaciones a probar
    cv=cv_strategy,        # estrategia de CV
    scoring='f1',          # metrica a maximizar
    n_jobs=-1,             # paralelizar
    verbose=1              # mostrar progreso
)

# TODO: Ejecutar la busqueda
# .fit() prueba todas las combinaciones y guarda la mejor
___  # grid_search.fit(X_temp, y_temp)

print("\nBusqueda completada!")


In [ ]:
# ============================================================
# BLOQUE: Ver Mejores Parametros
# ============================================================

# .best_params_ contiene el diccionario de los mejores hiperparametros
print("Mejores hiperparametros encontrados:")
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")

# .best_score_ contiene el mejor score promedio de CV
print(f"\nMejor F1 Score (CV): {grid_search.best_score_:.4f}")

# .best_estimator_ es el modelo ya entrenado con los mejores params
mejor_dt = grid_search.best_estimator_
print(f"\nMejor modelo: {mejor_dt}")

# Ver los top 5 resultados
resultados = pd.DataFrame(grid_search.cv_results_)
top5 = resultados.sort_values('rank_test_score').head(5)
cols_mostrar = ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
print("\nTop 5 combinaciones:")
print(top5[cols_mostrar].to_string(index=False))


In [ ]:
# ============================================================
# BLOQUE: Curva de Aprendizaje
# ============================================================
# La curva de aprendizaje muestra como cambia el score
# a medida que aumenta el tamano del conjunto de entrenamiento
#
# DIAGNOSTICO:
#   OVERFITTING: train score alto, val score bajo
#     Solucion: mas datos, regularizacion, modelo mas simple
#   UNDERFITTING: ambos scores bajos
#     Solucion: modelo mas complejo, mas features
#   BUEN FIT: ambos scores altos y cercanos

# learning_curve calcula scores para diferentes tamanos de train
# train_sizes: proporciones del dataset a usar
train_sizes, train_scores, val_scores = learning_curve(
    mejor_dt,              # modelo a evaluar (ya optimizado)
    X_temp,                # features
    y_temp,                # target
    train_sizes=np.linspace(0.1, 1.0, 10),  # 10 puntos de 10% a 100%
    cv=cv_strategy,        # cross-validation
    scoring='f1',          # metrica
    n_jobs=-1              # paralelizar
)

# Calcular media y desviacion estandar de los scores
train_mean = train_scores.mean(axis=1)  # promedio por tamano
train_std  = train_scores.std(axis=1)   # std por tamano
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

# Graficar
fig, ax = plt.subplots(figsize=(9, 6))

# Curva de entrenamiento
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Train score')
ax.fill_between(train_sizes,
                train_mean - train_std,
                train_mean + train_std,
                alpha=0.15, color='steelblue')  # banda de incertidumbre

# Curva de validacion
ax.plot(train_sizes, val_mean, 'o-', color='darkorange', label='Validation score')
ax.fill_between(train_sizes,
                val_mean - val_std,
                val_mean + val_std,
                alpha=0.15, color='darkorange')

ax.set_xlabel('Tamano del conjunto de entrenamiento', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Curva de Aprendizaje - Mejor Decision Tree', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"\nDiagnostico:")
print(f"  Train score final: {train_mean[-1]:.4f}")
print(f"  Val score final:   {val_mean[-1]:.4f}")
gap = train_mean[-1] - val_mean[-1]
if gap > 0.1:
    print(f"  Gap = {gap:.4f} --> posible OVERFITTING")
elif val_mean[-1] < 0.6:
    print(f"  Scores bajos --> posible UNDERFITTING")
else:
    print(f"  Gap = {gap:.4f} --> modelo bien ajustado")


In [ ]:
# ============================================================
# BLOQUE: Evaluacion Final en Holdout Set
# ============================================================
# Ahora si podemos usar el holdout set (datos que NUNCA vio el modelo)
# Esta es la evaluacion mas honesta del modelo

# Predecir en el holdout set con el mejor modelo
# mejor_dt ya fue entrenado con X_temp completo
y_pred_holdout = mejor_dt.predict(X_holdout)

# Calcular metricas finales
acc_final = accuracy_score(y_holdout, y_pred_holdout)
f1_final  = f1_score(y_holdout, y_pred_holdout, zero_division=0)

print("=" * 50)
print("EVALUACION FINAL EN HOLDOUT SET")
print("=" * 50)
print(f"  Accuracy: {acc_final:.4f}")
print(f"  F1 Score: {f1_final:.4f}")
print("=" * 50)

print("\nClassification Report completo:")
print(classification_report(y_holdout, y_pred_holdout, target_names=['No Churn', 'Churn']))


## Resumen de la Clase 11

### Checklist:
- [ ] Entender por que un solo split es insuficiente
- [ ] Aplicar `cross_val_score` con `StratifiedKFold`
- [ ] Visualizar scores por fold
- [ ] Construir un `Pipeline` con scaler + modelo
- [ ] Evaluar Pipeline con cross-validation
- [ ] Usar `GridSearchCV` para optimizar hiperparametros
- [ ] Interpretar curva de aprendizaje (overfitting/underfitting)
- [ ] Evaluar modelo final en holdout set

### Conceptos clave:
```
Data Leakage: cuando el modelo ve datos de test durante el entrenamiento
  -> Pipeline previene esto

Overfitting: train >> val  -> simplificar modelo, mas datos, regularizacion
Underfitting: ambos bajos -> modelo mas complejo, mas features
```

### Proxima clase:
Clase 12 - Proyecto Final: integrar todo lo aprendido en un analisis completo
